# Differentiation Matrix Validation for Nodal DG Method

Consolidated polynomial exactness validation for differentiation matrices.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.core.generators import build_nodes
from src.core.render_utils import VERTICES_2D, VERTICES_3D
from src.bases import vandermonde_2d_dubiner, grad_vandermonde_2d_dubiner
from src.reconstruction import build_differentiation_matrices


def get_reference_data(method: str, k: int):
    nodes = build_nodes(method, k, VERTICES_2D, VERTICES_3D)
    bary_coords = np.array([n.barycentric for n in nodes], dtype=float)
    weights = np.array([n.weight for n in nodes], dtype=float)
    xi = 2.0 * bary_coords[:, 2] - 1.0
    eta = 2.0 * bary_coords[:, 1] - 1.0
    return {
        "nodes": nodes,
        "bary_coords": bary_coords,
        "weights": weights,
        "xi": xi,
        "eta": eta,
    }


def exponent_pairs(total_degree: int):
    return [(i, total_degree - i) for i in range(total_degree + 1)]

In [2]:
METHODS = ["table1", "table2"]
K_VALUES = [1, 2, 3, 4]
TOLERANCE_DIFF = 1e-13

In [3]:
rows = []
overall_passed = True

for method in METHODS:
    for k in K_VALUES:
        ref = get_reference_data(method, k)
        xi = ref["xi"]
        eta = ref["eta"]

        V = vandermonde_2d_dubiner(xi, eta, k)
        V_xi, V_eta = grad_vandermonde_2d_dubiner(xi, eta, k)
        D_xi, D_eta = build_differentiation_matrices(V, V_xi, V_eta, w=ref["weights"])

        max_degree = k

        for degree in range(max_degree + 1):
            for i, j in exponent_pairs(degree):
                f = (xi ** i) * (eta ** j)
                df_dxi_exact = np.zeros_like(xi) if i == 0 else i * (xi ** (i - 1)) * (eta ** j)
                df_deta_exact = np.zeros_like(eta) if j == 0 else j * (xi ** i) * (eta ** (j - 1))

                df_dxi_num = D_xi @ f
                df_deta_num = D_eta @ f

                max_dxi_error = float(np.max(np.abs(df_dxi_num - df_dxi_exact)))
                max_deta_error = float(np.max(np.abs(df_deta_num - df_deta_exact)))

                is_exact = (max_dxi_error < TOLERANCE_DIFF) and (max_deta_error < TOLERANCE_DIFF)
                if not is_exact:
                    overall_passed = False

                rows.append({
                    "method": method,
                    "k": k,
                    "max_degree": max_degree,
                    "poly_degree": degree,
                    "i": i,
                    "j": j,
                    "term": f"xi^{i} * eta^{j}",
                    "max_abs_dxi_error": max_dxi_error,
                    "max_abs_deta_error": max_deta_error,
                    "status": "✓ PASS" if is_exact else "✗ FAIL",
                })

diff_df = pd.DataFrame(rows)
display(diff_df)

,method,k,max_degree,poly_degree,i,j,term,max_abs_dxi_error,max_abs_deta_error,status
0,table1,1,1,0,0,0,xi^0 * eta^0,0.000000e+00,1.110223e-16,✓ PASS
1,table1,1,1,1,0,1,xi^0 * eta^1,0.000000e+00,0.000000e+00,✓ PASS
2,table1,1,1,1,1,0,xi^1 * eta^0,2.220446e-16,2.220446e-16,✓ PASS
3,table1,2,2,0,0,0,xi^0 * eta^0,4.440892e-16,3.330669e-16,✓ PASS
4,table1,2,2,1,0,1,xi^0 * eta^1,2.220446e-16,5.551115e-16,✓ PASS
...,...,...,...,...,...,...,...,...,...,...
63,table2,4,4,4,0,4,xi^0 * eta^4,1.332268e-15,8.881784e-16,✓ PASS
64,table2,4,4,4,1,3,xi^1 * eta^3,8.881784e-16,1.554312e-15,✓ PASS
65,table2,4,4,4,2,2,xi^2 * eta^2,1.998401e-15,1.776357e-15,✓ PASS
66,table2,4,4,4,3,1,xi^3 * eta^1,6.661338e-16,8.881784e-16,✓ PASS


In [4]:
summary_df = (
    diff_df.groupby(["method", "k", "status"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
display(summary_df)

worst_df = (
    diff_df.groupby(["method", "k"], as_index=False)
    .agg({
        "max_abs_dxi_error": "max",
        "max_abs_deta_error": "max",
    })
)
display(worst_df)

if overall_passed:
    print("✓ SUCCESS! Differentiation matrices are exact up to configured polynomial limits.")
else:
    print("✗ FAILURE! Some polynomial exactness cases failed for differentiation matrices.")

,method,k,status,count
0,table1,1,✓ PASS,3
1,table1,2,✓ PASS,6
2,table1,3,✓ PASS,10
3,table1,4,✓ PASS,15
4,table2,1,✓ PASS,3
5,table2,2,✓ PASS,6
6,table2,3,✓ PASS,10
7,table2,4,✓ PASS,15


,method,k,max_abs_dxi_error,max_abs_deta_error
0,table1,1,2.220446e-16,2.220446e-16
1,table1,2,4.440892e-16,6.661338e-16
2,table1,3,1.110223e-15,2.220446e-15
3,table1,4,3.108624e-15,3.552714e-15
4,table2,1,2.220446e-16,2.220446e-16
5,table2,2,9.020562e-16,8.881784e-16
6,table2,3,8.881784e-16,1.332268e-15
7,table2,4,2.664535e-15,3.885781e-15


✓ SUCCESS! Differentiation matrices are exact up to configured polynomial limits.


In [ ]:
# ==========================================
# Spectral Convergence Test (平滑函數收斂性測試)
# 測試函數: f(xi, eta) = sin(pi * xi) * cos(pi * eta)
# ==========================================

import numpy as np
import pandas as pd
from IPython.display import display

results = []

for method in ["table1", "table2"]:
    prev_L2 = None
    prev_Linf = None

    for k in [1, 2, 3, 4]:
        # 1. 獲取節點與矩陣 (沿用您先前的 Helper function)
        ref = get_reference_data(method, k)
        xi = ref["xi"]
        eta = ref["eta"]
        weights = ref["weights"]

        V = vandermonde_2d_dubiner(xi, eta, k)
        V_xi, V_eta = grad_vandermonde_2d_dubiner(xi, eta, k)
        D_xi, D_eta = build_differentiation_matrices(V, V_xi, V_eta, w=weights)

        # 2. 定義平滑測試函數與其解析導數
        f = np.sin(0.1 * np.pi * xi) * np.cos(0.1 * np.pi * eta)
        df_dxi_exact = 0.1 * np.pi * np.cos(0.1 * np.pi * xi) * np.cos(0.1 * np.pi * eta)
        df_deta_exact = -0.1 * np.pi * np.sin(0.1 * np.pi * xi) * np.sin(0.1 * np.pi * eta)

        # 3. 計算數值導數
        df_dxi_num = D_xi @ f
        df_deta_num = D_eta @ f

        # 4. 計算誤差 (利用積分權重計算連續 L2 Norm 近似值)
        err_xi = df_dxi_num - df_dxi_exact
        err_eta = df_deta_num - df_deta_exact

        Linf = max(np.max(np.abs(err_xi)), np.max(np.abs(err_eta)))
        L2 = np.sqrt(np.sum(weights * (err_xi**2 + err_eta**2)))

        # 5. 計算收斂率 (Convergence Rate)
        if prev_L2 is not None and prev_L2 > 0:
            # 代數收斂率公式：log(E_old / E_new) / log(k_new / k_old)
            cr_L2 = np.log(prev_L2 / L2) / np.log(k / (k - 1))
            cr_Linf = np.log(prev_Linf / Linf) / np.log(k / (k - 1))
        else:
            cr_L2 = np.nan
            cr_Linf = np.nan

        # 6. 儲存結果
        results.append({
            "Table": method.capitalize(),
            "k": k,
            "L2": L2,
            "C.R.(L2)": cr_L2,
            "Linf": Linf,
            "C.R.(Linf)": cr_Linf
        })

        prev_L2 = L2
        prev_Linf = Linf

# 建立 DataFrame
df_spectral = pd.DataFrame(results)

# 格式化顯示 (確保科學記號與小數位數對齊)
df_spectral["L2"] = df_spectral["L2"].apply(lambda x: f"{x:.4e}")
df_spectral["Linf"] = df_spectral["Linf"].apply(lambda x: f"{x:.4e}")
df_spectral["C.R.(L2)"] = df_spectral["C.R.(L2)"].apply(lambda x: "-" if pd.isna(x) else f"{x:.2f}")
df_spectral["C.R.(Linf)"] = df_spectral["C.R.(Linf)"].apply(lambda x: "-" if pd.isna(x) else f"{x:.2f}")

display(df_spectral)

,Table,k,L2,C.R.(L2),Linf,C.R.(Linf)
0,Table1,1,1.7357e-02,-,1.9206e-02,-
1,Table1,2,1.2116e-02,0.52,2.6646e-02,-0.47
2,Table1,3,1.1283e-04,11.53,4.0122e-04,10.35
3,Table1,4,8.1675e-05,1.12,4.2649e-04,-0.21
4,Table2,1,1.0619e-02,-,1.0232e-02,-
5,Table2,2,1.1160e-02,-0.07,2.2071e-02,-1.11
6,Table2,3,8.3640e-05,12.07,1.7847e-04,11.88
7,Table2,4,7.4992e-05,0.38,2.4234e-04,-1.06
